In [4]:
import requests
import json
import pandas as pd
import base64
import os
import io

In [5]:
# pull Road Closure feed and create road_events_json object

# un-comment out the following lines when running in VS Code, comment out those below
password = pd.read_csv("passwords.csv")
password_nps = password["password"][0]

# comment out the above when running in DataBricks, un-comment below
# password_nps = dbutils.secrets.get(scope="volpe-npstravelerinfo-dev", key="NPS API Key")

road_events_url = "https://developer.nps.gov/api/v1/roadevents?api_key=" + password_nps

response_API = requests.get(road_events_url)
data = response_API.text
road_events_json = json.loads(data)


In [6]:
# pull Alerts feed and create alerts_json object
alerts_url = "https://developer.nps.gov/api/v1/alerts?limit=10000&api_key=" + password_nps

response_API = requests.get(alerts_url)
data = response_API.text
alerts_json = json.loads(data)


In [11]:
# make a dataframe from the data sources section of Road Events
data_sources_df = pd.json_normalize(
    road_events_json, 
    record_path=['road_event_feed_info','data_sources'])

# make a dataframe from the 'properties' section of Road Events
properties_list = [f['properties'] for f in road_events_json['features']]
properties_df = pd.json_normalize(properties_list)


# merge the two dataframes on the 'data_source_id' field
road_closures_feed_df = pd.merge(
    properties_df,                   # The main data (left)
    data_sources_df,                    # The metadata (right)
    left_on='core_details.data_source_id', 
    right_on='data_source_id',
    how='left'                     # Ensures we don't lose any road events
)

# add a column for today's date and make it the first column
today = pd.Timestamp.now().normalize()
road_closures_feed_df.insert(0, 'extraction_date', today)


### save dataframe as a csv with today's date in the file name
today_str = today.strftime('%Y-%m-%d')

# TODO: keep this uncommented until we get saving to sharepoint to work. After it works, comment out.
road_closures_feed_df.to_csv('Road_Closure_Feeds/' + today_str + '_Road_Closure_Feed.csv', index=False)

# Power Automate URL
power_automate_url = r"https://prod-06.usgovtexas.logic.azure.us:443/workflows/311b43307bf146a08864bfed85521acf/triggers/manual/paths/invoke?api-version=2016-06-01&sp=%2Ftriggers%2Fmanual%2Frun&sv=1.0&sig=dd3VSayLwdkoloVBbJMKzzNVfDwre-Qu3UizEcVQ2I4"

file_path = today_str + '_Road_Closure_Feed.csv'
file_name = os.path.basename(file_path)

# Convert DataFrame to a CSV string (instead of a local file)
csv_buffer = io.StringIO()
road_closures_feed_df.to_csv(csv_buffer, index=False)
csv_string = csv_buffer.getvalue()

# Encode the string to Base64 (Power Automate handles Base64 best for files)
# We encode the string to bytes, then base64, then back to a utf-8 string for the JSON
base64_file = base64.b64encode(csv_string.encode('utf-8')).decode('utf-8')

# Prepare the payload
file_name = f"{today_str}_Road_Closure_Feed.csv"
payload = {
    "fileName": file_name,
    "fileContent": base64_file
}

# Send to Power Automate
response = requests.post(power_automate_url, json=payload)

if response.status_code == 202:
    print("Success! File sent to SharePoint.")
else:
    print(f"Failed with status: {response.status_code}")

### save the raw json too
# TODO: keep this uncommented until we get saving to sharepoint to work. After it works, comment out.
with open( 'Road_Closure_Feeds/' + today_str + '_road_closures_feed.json', 'w', encoding='utf-8') as f:
    json.dump(road_events_json, f, indent=4)
    
# Convert JSON object to a pretty-printed string
json_string = json.dumps(road_events_json, indent=4)

# Encode the string to Base64
# turn the string to bytes, encode, then back to a utf-8 string for the JSON payload
base64_json = base64.b64encode(json_string.encode('utf-8')).decode('utf-8')

# Prepare the payload
file_name = f"{today_str}_road_closures_feed.json"
payload = {
    "fileName": file_name,
    "fileContent": base64_json
}

# Send to Power Automate
response = requests.post(power_automate_url, json=payload)

if response.status_code == 202:
    print(f"Successfully uploaded {file_name} to SharePoint!")
else:
    print(f"Failed: {response.status_code} - {response.text}")



Success! File sent to SharePoint.
Successfully uploaded 2026-04-08_road_closures_feed.json to SharePoint!


In [12]:
base64_file

'...'

In [0]:
# make a dataframe out of the Alerts feed
alerts_df = pd.json_normalize(alerts_json, record_path=['data'])

# add a column for today's date and make it the first column
today = pd.Timestamp.now().normalize()
alerts_df.insert(0, 'extraction_date', today)

### save it as a csv with today's date in the file name
# TODO: keep this uncommented until we get saving to sharepoint to work. After it works, comment out.
alerts_df.to_csv('Alerts_Feeds/' + today_str + '_Alerts_Feed.csv', index=False)

file_path = today_str + '_alerts_feed.csv'
file_name = os.path.basename(file_path)

# Convert DataFrame to a CSV string (instead of a local file)
csv_buffer = io.StringIO()
alerts_df.to_csv(csv_buffer, index=False)
csv_string = csv_buffer.getvalue()

# Encode the string to Base64 (Power Automate handles Base64 best for files)
# We encode the string to bytes, then base64, then back to a utf-8 string for the JSON
base64_file = base64.b64encode(csv_string.encode('utf-8')).decode('utf-8')

# Prepare the payload
file_name = f"{today_str}_alerts_feed.csv"
payload = {
    "fileName": file_name,
    "fileContent": base64_file
}

# Send to Power Automate
# ultimately ends up in this sharepoint folder via Power Automate: https://usdot.sharepoint.com/teams/volpe-proj-VU16A100/Shared%20Documents/Forms/AllItems.aspx?id=%2Fteams%2Fvolpe%2Dproj%2DVU16A100%2FShared%20Documents%2FResearch%20and%20Innovation%2FSubgroup%20Support%2FTraveler%20Information%20Technologies%2FRoads%2FDaily%20Feed%20Pull&sortField=LinkFilename&isAscending=false&viewid=b9fcbbd4%2Da7fd%2D4879%2Db80c%2D93ea59e50f1a&csf=1&CID=af4918fd%2D2199%2D4a6b%2Da73a%2Dceab3b298bbc&FolderCTID=0x012000FE7428814DF42F40AC105097781CC61B
response = requests.post(power_automate_url, json=payload)

if response.status_code == 202:
    print("Success! File sent to SharePoint.")
else:
    print(f"Failed with status: {response.status_code}")

### save the raw json too
# TODO: keep this uncommented until we get saving to sharepoint to work. After it works, comment out.
with open( 'Alerts_Feeds/' + today_str + '_alerts_feed.json', 'w', encoding='utf-8') as f:
    json.dump(alerts_json, f, indent=4)
    
# Convert JSON object to a pretty-printed string
json_string = json.dumps(alerts_json, indent=4)

# Encode the string to Base64
# turn the string to bytes, encode, then back to a utf-8 string for the JSON payload
base64_json = base64.b64encode(json_string.encode('utf-8')).decode('utf-8')

# Prepare the payload
file_name = f"{today_str}_alerts_feed.json"
payload = {
    "fileName": file_name,
    "fileContent": base64_json
}

# Send to Power Automate
response = requests.post(power_automate_url, json=payload)

if response.status_code == 202:
    print(f"Successfully uploaded {file_name} to SharePoint!")
else:
    print(f"Failed: {response.status_code} - {response.text}")